# Notebook 17 — From Base Model to Instruction, Reasoning, and Tool-Using Model

    ## Learning objectives

    - Map continued pretraining, SFT, preference optimization, RL, distillation, and safety stages
- Represent the data and objective used at each post-training stage
- Explain reasoning models, test-time compute, tool use, and capability evaluation without mystique

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = ['transformers>=4.51,<5', 'datasets>=3.5,<6', 'trl>=0.16', 'peft>=0.15']

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 17.1 Post-training changes behavior, not the fundamental token interface

A base causal model predicts plausible continuations. A user-facing assistant must interpret
roles, follow requests, respect policies, call tools, express uncertainty, and stop correctly.
Post-training reshapes which continuations are likely under those contexts. Most stages still
operate through token log probabilities; what changes is the data source, target, reward, or
optimization constraint. Instruction tuning is therefore not a switch from “next-token model”
to a different species of system.

There is no single mandatory recipe. A common pathway is base pretraining → optional domain
continued pretraining → supervised instruction tuning → preference optimization or RL → safety
tuning and adversarial evaluation → task-specific adapters, tool training, distillation, and
deployment optimization. Teams often iterate, mix data, or branch checkpoints. Every stage can
improve a target while degrading calibration, diversity, safety, or general capability.


In [ ]:
stages = [
    ("base pretraining", "raw token sequences", "next-token cross-entropy", "broad representations"),
    ("continued pretraining", "domain documents", "next-token cross-entropy", "domain adaptation"),
    ("SFT", "prompt + desired response", "masked next-token cross-entropy", "instruction behavior"),
    ("preference optimization", "chosen/rejected responses", "relative policy objective", "human preferences"),
    ("RL with verifiers", "sampled trajectories + rewards", "policy optimization", "reasoning/search behavior"),
    ("distillation", "teacher outputs/soft targets", "CE or divergence", "compress capabilities"),
]
for name, data, objective, purpose in stages:
    print(f"{name:24} | {data:29} | {objective:28} | {purpose}")


## 17.2 Data contracts for different objectives

SFT demonstrations answer “what should the model produce?” Preference pairs answer “which of
these is better?” Reward-model records attach scalar or categorical judgments. Online RL records
prompts, sampled trajectories, rewards, and old-policy probabilities. Tool training additionally
needs exact schemas, valid arguments, tool results, recovery examples, and permission boundaries.
Quality depends on coverage and labeling consistency more than on the beauty of a trainer call.


In [ ]:
examples = {
    "sft": {"messages": [{"role": "user", "content": "Return JSON."},
                          {"role": "assistant", "content": '{"ok": true}'}]},
    "preference": {"prompt": "Explain entropy.", "chosen": "Entropy measures uncertainty...",
                   "rejected": "Entropy is always disorder."},
    "verifiable_reasoning": {"prompt": "What is 17*23?", "response": "391", "reward": 1.0,
                             "verifier": "exact_answer_v2"},
    "tool_trajectory": {"messages": [{"role": "user", "content": "Weather in Boston?"}],
                        "tool_call": {"name": "weather", "arguments": {"city": "Boston"}},
                        "tool_result": {"temperature_c": 21}, "final": "It is 21°C."},
}
for objective, record in examples.items(): print("\n", objective, record)


In [ ]:
# Completion-only SFT: prompt tokens provide context but are not prediction targets.
import torch
prompt_ids = torch.tensor([11, 12, 13, 14])
answer_ids = torch.tensor([21, 22, 23])
input_ids = torch.cat((prompt_ids, answer_ids))
labels = input_ids.clone(); labels[:len(prompt_ids)] = -100
print("input IDs:", input_ids.tolist())
print("SFT labels:", labels.tolist(), "(-100 is ignored by cross-entropy)")


## 17.3 Preference learning, reward models, and RL

A reward model is a learned measurement instrument, not truth. It can inherit rater bias and
reward superficial length, confidence, or style. DPO directly increases a chosen response's
advantage over a rejected one relative to a reference policy. Online methods instead sample
from the current policy, score results, and optimize expected reward while constraining drift.
PPO is historically common; group-relative methods such as GRPO can compare several sampled
completions without a separate value model. The surrounding engineering—sampling diversity,
reward normalization, KL control, filtering, and held-out evaluation—is part of the algorithm.


In [ ]:
import torch.nn.functional as F
policy_margin = torch.tensor([0.8, -0.2, 1.4])
reference_margin = torch.tensor([0.1, 0.0, 0.5])
beta = 0.1
dpo_losses = -F.logsigmoid(beta * (policy_margin - reference_margin))
print("per-pair DPO losses:", dpo_losses.tolist(), "mean:", dpo_losses.mean().item())


## 17.4 What makes a “reasoning” or “thinking” model?

The label usually describes a training and inference regime, not a new transformer primitive.
Capabilities may be elicited through high-quality worked solutions, distilled traces from a
stronger teacher, rejection sampling, process supervision, and RL with verifiable outcome
rewards for math, code, or games. At inference, allocating more tokens, sampling multiple
candidates, voting, using search, or invoking a verifier spends test-time compute to improve
success probability. Gains are strongest where answers can be checked and weaker where rewards
are subjective or hackable.

A visible rationale is not guaranteed to be faithful to the model's internal computation, and
long answers are not synonymous with reasoning. Products may expose a concise explanation while
keeping internal scratch work private. Evaluate final correctness, robustness to perturbations,
calibration, token/latency cost, and verifier integrity rather than grading prose alone.


In [ ]:
# Test-time compute as best-of-N under an independent toy success model.
p_one = 0.35
for samples in [1, 2, 4, 8, 16]:
    at_least_one = 1 - (1 - p_one) ** samples
    print(f"samples={samples:2d} theoretical pass@N={at_least_one:.1%}")
print("Real samples are correlated and selection/verifiers are imperfect.")


## 17.5 Instruction following, tools, safety, and specialization

Chat templates serialize roles into tokens; the model must be trained on that exact convention.
Tool use is structured prediction plus an external control plane: the model proposes a call, but
trusted software validates schema, authorization, budgets, and side effects. Retrieval adds current
knowledge without placing it in weights. Adapters can specialize style or tasks cheaply. Safety
training combines desired refusals and safe completions with system-level isolation, monitoring,
red teaming, and incident response; weights alone cannot enforce authorization.

Release gates should compare each checkpoint with its parent on target capability, broad retention,
safety, calibration, latency, output length, and subgroup slices. Keep frozen prompts and blinded
human evaluation, but also use deterministic validators whenever answers are mechanically checkable.
The deployable model is the checkpoint plus tokenizer, chat template, decoding policy, tools,
retrieval, safety controls, and serving configuration.


In [ ]:
decision_guide = {
    "fresh/current facts": "retrieval or tools",
    "consistent response format": "SFT or constrained decoding",
    "cheap domain specialization": "LoRA/adapter SFT",
    "rank subjective answer quality": "preference data + DPO/reward modeling",
    "verifiable multi-step problem solving": "reasoning SFT + RL/verifier + test-time search",
    "latency/cost reduction": "distillation, quantization, serving optimization",
    "permission enforcement": "application control plane—not model training",
}
for need, method in decision_guide.items(): print(f"{need:38} -> {method}")


## 17.6 A capability stack rather than one magic checkpoint

Separate knowledge in weights from knowledge supplied in context, behaviors learned during tuning,
and guarantees enforced by software. Continued pretraining may internalize stable domain patterns;
retrieval supplies changing evidence; SFT teaches response conventions; preference optimization
shifts ambiguous quality judgments; tools execute exact or current operations; constrained decoding
enforces syntax; and the application control plane owns identity, permissions, budgets, and audit.
Asking training to solve a systems problem creates brittle assurances.

The stages also have distinct failure modes. Domain adaptation can forget general skills. SFT can
overfit a style and reduce diversity. Preference training can exploit judge shortcuts or make answers
verbose and overconfident. Verifier-based RL can reward-hack an incomplete checker. Distillation can
transfer teacher errors. Quantization can produce capability regressions on sensitive tasks. Tool
training can yield syntactically valid but unauthorized calls. Maintain stage-specific diagnostics
alongside end-to-end product evaluation.


In [ ]:
release_gates = {
    "continued pretraining": ["domain perplexity", "general retention", "memorization/privacy"],
    "SFT": ["instruction adherence", "format validity", "general retention"],
    "preference/RL": ["blinded preference", "reward hacking", "calibration", "safety"],
    "reasoning": ["pass@1", "pass@N", "verifier robustness", "tokens and latency"],
    "tools": ["call accuracy", "argument validity", "permission denial", "recovery"],
    "deployment": ["quality parity", "throughput", "tail latency", "resource saturation"],
}
for stage, gates in release_gates.items(): print(stage, "->", ", ".join(gates))


## 17.7 Choosing an optimization family

Use SFT when you have trusted target demonstrations. Use pairwise preference methods when producing
a single ideal answer is hard but comparison is reliable. Use reward modeling and online RL when the
policy must explore beyond a static set and the reward can be audited. Use verifiable RL where a
deterministic or independently trusted checker captures the real objective. Do not use RL merely
because a task sounds difficult: unstable rewards and weak evaluation can make it an expensive route
to worse behavior.

Reasoning data deserves special scrutiny. Correct final answers can accompany invalid intermediate
steps, and plausible traces can accompany wrong answers. Outcome supervision is scalable where final
answers are checkable; process supervision offers denser feedback but is costly and can encode one
preferred solution style. Synthetic traces multiply coverage but also correlated teacher errors.
Filter with independent checks, retain diverse strategies, and test on uncontaminated problems.

Finally, optimize the inference policy jointly with the model. Temperature, maximum thinking budget,
early stopping, number of candidates, tool limits, and selection method determine quality and cost.
Report accuracy against generated tokens and wall-clock latency—not accuracy alone—so a “better”
reasoning configuration does not conceal a tenfold serving bill.


## 17.8 Capability stages need separate gates

Pretraining, instruction tuning, preference optimization, reasoning reinforcement, tool training, and safety tuning optimize different data and objectives. Evaluate the input checkpoint before each stage and maintain a stable retention suite. A later stage can improve response style while damaging likelihood, calibration, multilingual ability, or domain knowledge. Define promotion gates by capability and risk slice, retain parent checkpoints, and use matched inference settings. Model merging or adapter composition is another intervention and needs its own evaluation.


In [ ]:
stages=[{"name":"base","format":.12,"knowledge":.72},{"name":"sft","format":.84,"knowledge":.70},{"name":"preference","format":.9,"knowledge":.66}]
for before,after in zip(stages,stages[1:]): print(before["name"],"->",after["name"],{k:after[k]-before[k] for k in ("format","knowledge")})


## 17.9 Reasoning traces versus outcomes

Models can learn from final answers, concise derivations, long rationales, process labels, verifier rewards, or tool trajectories. These supervise different behaviors. Long hidden or visible reasoning is not automatically faithful, and optimizing a final-answer verifier can create reward hacking. Evaluate correctness under fixed budgets, robustness to perturbations, calibration, verbosity, tool use, and outcome verification. Product interfaces should not depend on exposing unrestricted internal traces; request concise evidence or auditable tool receipts where appropriate.


In [ ]:
candidates=[{"answer":42,"tokens":12,"verified":True},{"answer":41,"tokens":120,"verified":False},{"answer":42,"tokens":95,"verified":True}]
valid=[x for x in candidates if x["verified"]]; print("shortest verified",min(valid,key=lambda x:x["tokens"]))


## Primary references and further study

Use the pinned library documentation that matches your environment. Papers explain the method and assumptions; current official documentation defines the executable API.

- [Training language models to follow instructions](https://arxiv.org/abs/2203.02155)
- [TRL documentation](https://huggingface.co/docs/trl/index)


## Exercises

    1. Design a staged recipe for a code assistant and define a release gate after every checkpoint.
2. Create five preference pairs where style conflicts with correctness; write adjudication rules.
3. Compare single-sample accuracy with best-of-N accuracy and total generated-token cost.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
